[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/modulo_4/10_Image_Embeddings.ipynb)

# Embeddings de imágenes — CLIP

En **07** y **08** vimos embeddings de palabras (Word2Vec), y en **09** embeddings de texto (Gemini). Aquí: **una imagen también se puede volver un vector**, con **CLIP** (OpenAI, 2021).

Lo que hace especial a CLIP: mete texto e imágenes en el **mismo espacio**. Por eso podemos comparar una frase contra una foto y que tenga sentido.

El plan es simple:

1. Vemos 8 fotos y les escribimos una **buena descripción** de lo que de verdad se ve en cada una.
2. Convertimos las fotos en vectores.
3. Le preguntamos algo al modelo (en texto) y nos regresa la **foto más parecida**.

**Nota:** esta es una primera versión del notebook; la iremos afinando.

## Archivos que necesitas

Ninguno. Las imágenes de ejemplo se descargan solas desde Wikimedia Commons.

## Setup — instalar librerías

`transformers` trae CLIP listo para usar; no hace falta entrenar nada.

In [ ]:
%pip install -q transformers torch pillow numpy matplotlib requests scikit-learn plotly

**GPU:** no es necesaria. El modelo (~150 MB) corre rápido en CPU. Si tienes GPU activada en Colab, se usa sola.

In [ ]:
import io

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import requests
import torch
from PIL import Image
from sklearn.decomposition import PCA
from transformers import CLIPModel, CLIPProcessor

plt.rcParams.update({"figure.dpi": 100, "axes.grid": False})

## 0 — Las 8 fotos, y lo que de verdad se ve en cada una

Antes de usar el modelo, vimos cada foto y escribimos qué hay en ella — no una etiqueta genérica tipo "a photo of a X", sino lo que realmente se ve (de qué color, qué está haciendo, dónde está). Esa descripción es la que le vamos a dar a CLIP.

In [ ]:
# Wikimedia bloquea el User-Agent por defecto de requests (403 Forbidden);
# pide uno descriptivo — https://meta.wikimedia.org/wiki/User-Agent_policy
HEADERS_WIKIMEDIA = {"User-Agent": "IberoDiplomadoNLP/1.0 (uso educativo, sin fines comerciales)"}

IMAGENES = {
    "gato": {
        "url": "https://commons.wikimedia.org/wiki/Special:FilePath/Cat03.jpg?width=400",
        "descripcion": "a close-up of an orange tabby cat with its head tilted, looking at the camera",
    },
    "perro": {
        "url": "https://commons.wikimedia.org/wiki/Special:FilePath/Golde33443.jpg?width=400",
        "descripcion": "a white golden retriever puppy lying on a mossy log in a garden",
    },
    "bicicleta": {
        "url": "https://commons.wikimedia.org/wiki/Special:FilePath/Bicycle.jpg?width=400",
        "descripcion": "two old rusty bicycles parked against a weathered wall",
    },
    "motocicleta": {
        "url": "https://commons.wikimedia.org/wiki/Special:FilePath/Motorcycle.jpg?width=400",
        "descripcion": "a motocross rider doing an aerial jump on a dirt bike at night, blue-toned action photo",
    },
    "pizza": {
        "url": "https://commons.wikimedia.org/wiki/Special:FilePath/Eq_it-na_pizza-margherita_sep2005_sml.jpg?width=400",
        "descripcion": "a whole margherita pizza with melted cheese and basil leaves on a white plate",
    },
    "hamburguesa": {
        "url": "https://commons.wikimedia.org/wiki/Special:FilePath/Cheeseburger.jpg?width=400",
        "descripcion": "a cheeseburger with lettuce on a sesame seed bun, studio photo on a white background",
    },
    "hot dog (comida)": {
        "url": "https://commons.wikimedia.org/wiki/Special:FilePath/Chicago-style_hot_dog.jpg?width=400",
        "descripcion": "a loaded hot dog sandwich with tomato, onion, pickle and mustard, wrapped in paper",
    },
    "paisaje de montaña": {
        "url": "https://commons.wikimedia.org/wiki/Special:FilePath/Mountain_landscape.jpg?width=400",
        "descripcion": "a misty mountain ridge with a lone hiker walking on a grassy trail",
    },
}


def descargar_imagen(url):
    """Baja una imagen y la abre con PIL, sin guardarla a disco."""
    resp = requests.get(url, headers=HEADERS_WIKIMEDIA, timeout=20)
    resp.raise_for_status()
    return Image.open(io.BytesIO(resp.content)).convert("RGB")


fotos = {nombre: descargar_imagen(info["url"]) for nombre, info in IMAGENES.items()}
nombres_fotos = list(fotos.keys())
descripciones = [IMAGENES[nombre]["descripcion"] for nombre in nombres_fotos]

fig, axes = plt.subplots(2, 4, figsize=(14, 7.5))
for ax, nombre in zip(axes.flat, nombres_fotos):
    ax.imshow(fotos[nombre])
    ax.set_title(nombre, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

for nombre in nombres_fotos:
    descripcion = IMAGENES[nombre]["descripcion"]
    print(f"{nombre:>20s}: {descripcion}")

## 1 — Cargar CLIP

`openai/clip-vit-base-patch32`: un encoder de imágenes (Vision Transformer) y un encoder de texto, entrenados juntos para que la imagen y su descripción caigan cerca en el mismo espacio de 512 dimensiones.

In [ ]:
MODELO_CLIP = "openai/clip-vit-base-patch32"

modelo = CLIPModel.from_pretrained(MODELO_CLIP)
procesador = CLIPProcessor.from_pretrained(MODELO_CLIP)

print(f"Modelo listo: {MODELO_CLIP}")

## 2 — Convertir las 8 fotos en vectores

Le pasamos las 8 imágenes a CLIP y obtenemos un vector de 512 números por cada una. `.pooler_output` es por cómo `transformers` entrega el resultado desde la versión 5 — antes regresaba el vector directo.

In [ ]:
entradas_imgs = procesador(images=list(fotos.values()), return_tensors="pt")
with torch.no_grad():
    vectores_imgs = modelo.get_image_features(**entradas_imgs).pooler_output

print(f"Shape: {vectores_imgs.shape}  (8 fotos, 512 dimensiones cada una)")


def similitud_coseno(u, v):
    u, v = u.detach().numpy(), v.detach().numpy()
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

## 3 — Pregúntale algo al modelo

Corre la celda de abajo una vez (define la función `buscar_imagen`). Después, para probar, solo necesitas **una línea**: `buscar_imagen("lo que se te ocurra")`. Convierte tu frase en un vector y la compara contra las 8 fotos — te regresa las 3 más parecidas.

**Nota:** CLIP funciona mejor en inglés; en español puede salir raro — es justo lo que estamos probando. Prueba algo que no sea ninguna de las 8 categorías, como `"patín"`.

In [ ]:
def buscar_imagen(frase, top=3):
    """Regresa (y dibuja) las imágenes más parecidas a la frase, de las que ya tenemos."""
    entrada = procesador(text=[frase], return_tensors="pt", padding=True)
    with torch.no_grad():
        vector = modelo.get_text_features(**entrada).pooler_output[0]

    ranking = sorted(
        ((nombre, similitud_coseno(vector, vi)) for nombre, vi in zip(nombres_fotos, vectores_imgs)),
        key=lambda x: -x[1],
    )[:top]

    fig, axes = plt.subplots(1, top, figsize=(4 * top, 4))
    for ax, (nombre, sim) in zip(axes, ranking):
        ax.imshow(fotos[nombre])
        ax.set_title(f"{nombre}\nsim = {sim:.3f}")
        ax.axis("off")
    plt.suptitle(f'Búsqueda: "{frase}"')
    plt.tight_layout()
    plt.show()

In [ ]:
buscar_imagen("cheese")

## 4 — Buscar por imagen (sin texto)

El modelo no sabe la diferencia entre "una pregunta en texto" y "una pregunta en foto" — las dos son solo un vector. Aquí metes **otra imagen**, distinta a las 8 que ya tenemos, y vemos a cuál de las 8 se parece más.

Puedes usar:
- una **URL** de internet (empieza con `http`), o
- el nombre de un archivo que **subiste al panel de archivos de Colab** (ícono de carpeta 📁 en la barra lateral izquierda) — cae directo en `/content/`, así que solo pones el nombre, ej. `"mi_foto.jpg"`.

In [ ]:
def cargar_imagen_externa(ruta_o_url):
    """Acepta una URL de internet o un archivo subido a Colab (en /content/)."""
    if ruta_o_url.startswith("http"):
        resp = requests.get(ruta_o_url, headers=HEADERS_WIKIMEDIA, timeout=20)
        resp.raise_for_status()
        return Image.open(io.BytesIO(resp.content)).convert("RGB")
    return Image.open(ruta_o_url).convert("RGB")


def imagenes_parecidas(ruta_o_url, top=3):
    """De nuestras 8 fotos, regresa las más parecidas a una imagen nueva."""
    imagen_nueva = cargar_imagen_externa(ruta_o_url)
    entrada = procesador(images=imagen_nueva, return_tensors="pt")
    with torch.no_grad():
        vector_nuevo = modelo.get_image_features(**entrada).pooler_output[0]

    ranking = sorted(
        ((nombre, similitud_coseno(vector_nuevo, vi)) for nombre, vi in zip(nombres_fotos, vectores_imgs)),
        key=lambda x: -x[1],
    )[:top]

    fig, axes = plt.subplots(1, top + 1, figsize=(4 * (top + 1), 4))
    axes[0].imshow(imagen_nueva)
    axes[0].set_title("tu imagen")
    axes[0].axis("off")
    for ax, (nombre, sim) in zip(axes[1:], ranking):
        ax.imshow(fotos[nombre])
        ax.set_title(f"{nombre}\nsim = {sim:.3f}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# Otra foto de gente patinando — para comparar contra el experimento de la sección 3,
# donde buscamos "patín" solo con texto y ganó "gato" (iba mal). Aquí con una imagen real:
imagenes_parecidas("https://cdn.cosmos.so/04a11635-f203-4b9b-893f-51cdc4e1ec58?format=webp")

## 5 — Buscar con imagen **y** texto combinados

También puedes combinar los dos: promedias el vector de una imagen nueva con el vector de una frase, y usas ese vector combinado para buscar entre nuestras 8 fotos. Es como pedir "una foto como esta, pero más X".

Mismo truco que en la sección 4 para la imagen: URL de internet, o un archivo que subiste a Colab.

`peso_texto` controla qué tanto pesa la frase (0 = ignora la frase, 1 = ignora la foto). Con 0.5 el cambio casi no se nota; con 0.7 sí.

In [ ]:
def buscar_imagen_y_texto(ruta_o_url, frase, peso_texto=0.7, top=3):
    """Combina una imagen nueva con una frase, y busca entre nuestras 8 fotos."""
    imagen_nueva = cargar_imagen_externa(ruta_o_url)
    entrada_img = procesador(images=imagen_nueva, return_tensors="pt")
    with torch.no_grad():
        vector_imagen = modelo.get_image_features(**entrada_img).pooler_output[0]

    entrada_texto = procesador(text=[frase], return_tensors="pt", padding=True)
    with torch.no_grad():
        vector_frase = modelo.get_text_features(**entrada_texto).pooler_output[0]

    vector_combinado = (1 - peso_texto) * vector_imagen + peso_texto * vector_frase

    ranking = sorted(
        ((nombre, similitud_coseno(vector_combinado, vi)) for nombre, vi in zip(nombres_fotos, vectores_imgs)),
        key=lambda x: -x[1],
    )[:top]

    fig, axes = plt.subplots(1, top + 1, figsize=(4 * (top + 1), 4))
    axes[0].imshow(imagen_nueva)
    axes[0].set_title(f'tu imagen\n+ "{frase}"')
    axes[0].axis("off")
    for ax, (nombre, sim) in zip(axes[1:], ranking):
        ax.imshow(fotos[nombre])
        ax.set_title(f"{nombre}\nsim = {sim:.3f}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
buscar_imagen_y_texto("https://cdn.cosmos.so/04a11635-f203-4b9b-893f-51cdc4e1ec58?format=webp", "food")

## Bonus — mapa de dónde cae cada imagen

Las imágenes y sus descripciones viven en 512 dimensiones — no se pueden graficar tal cual. Igual que con Word2Vec en el notebook 08, usamos **PCA** para aplastarlas a 3 dimensiones, y graficamos las 8 fotos junto con sus 8 descripciones en el mismo mapa. Si CLIP funciona bien, cada foto cae cerca de su propia descripción.

In [ ]:
entradas_desc = procesador(text=descripciones, return_tensors="pt", padding=True)
with torch.no_grad():
    vectores_desc = modelo.get_text_features(**entradas_desc).pooler_output

combinados = np.vstack([vectores_imgs.numpy(), vectores_desc.numpy()])
coords = PCA(n_components=3, random_state=42).fit_transform(combinados)
coords_imgs = coords[:len(fotos)]
coords_desc = coords[len(fotos):]

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=coords_imgs[:, 0], y=coords_imgs[:, 1], z=coords_imgs[:, 2],
    mode="markers+text",
    marker=dict(size=6, color="#2a78d6"),
    text=[f"📷 {nombre}" for nombre in nombres_fotos],
    textposition="top center", textfont=dict(size=10),
    name="imagen",
))

fig.add_trace(go.Scatter3d(
    x=coords_desc[:, 0], y=coords_desc[:, 1], z=coords_desc[:, 2],
    mode="markers+text",
    marker=dict(size=4, color="#eb6834"),
    text=descripciones,
    textposition="bottom center", textfont=dict(size=8),
    name="descripción",
))

fig.update_layout(
    title="CLIP: fotos y descripciones en el mismo espacio (PCA 3D)",
    scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
    width=950, height=750,
)
fig.show()

## Cierre

Con el mismo truco de siempre (vectores + similitud coseno) comparamos **modalidades distintas**: una foto contra una frase. Esa es la idea detrás de la búsqueda "texto → imagen" (como en Google Fotos o Pinterest).

**Lo que vale la pena notar:**

- Una descripción fiel a lo que de verdad se ve en la foto da una similitud más alta que una etiqueta genérica tipo "a photo of a X".
- `hot dog (comida)` es una trampa de nombre: si el modelo solo mirara la palabra "dog" se confundiría con el perro. Pruébalo en la sección 3.
- `paisaje de montaña` no se parece a ninguna categoría de comida, vehículo o animal — buen caso para ver qué tan baja queda la similitud cuando de verdad no hay nada parecido.

**Límites de esta primera versión:**

- CLIP funciona mejor con texto en **inglés** — en español la similitud baja un poco.
- No hay entrenamiento aquí: CLIP ya viene entrenado, solo lo usamos.